# Building pandapower Networks from UKPN LTDS Data

This tutorial demonstrates how to fetch UK Power Networks' Long Term Development Statement (LTDS) data and use it with [pandapower](https://www.pandapower.org/) for power system analysis.

## Important Note about CIM Data

The **LTDS CIM dataset** is a "Shared" dataset that requires special access. To request access:
- Email: networkinsights@ukpowernetworks.co.uk
- Complete the Shared Data Request Form
- Register and login to the UKPN portal

**This tutorial uses publicly available LTDS tables** (Table 2a, 2b, 3a, 5) which are accessible via the standard API.

## What is CIM?

The **Common Information Model (CIM)** is an IEC standard (IEC 61970/61968) that provides a structured representation of power system data. It enables interoperable data exchange between Distribution Network Operators (DNOs), software systems, and network modelling tools.

UKPN publishes LTDS data in CIM format, which includes:
- Network topology (buses, lines, transformers)
- Equipment specifications
- Connectivity information

## What is pandapower?

[pandapower](https://www.pandapower.org/) is an open-source Python library for power system modeling, analysis, and optimization. It combines the data analysis capabilities of pandas with power system calculation functions.

## Prerequisites

```bash
pip install ukpyn pandapower
```

## 1. Fetch LTDS Data from UKPN

First, let's use the `ltds` orchestrator to fetch data from the UK Power Networks Open Data Portal. We'll use publicly available LTDS tables (not CIM, which requires special access).

In [ ]:
import pandas as pd

from ukpyn import ltds

# Check available LTDS datasets
print("Available LTDS datasets:")
print(ltds.available_datasets)

# Expected output:
# Available LTDS datasets:
# ['table_2a', 'transformer_2w', 'table_2b', 'transformer_3w', 'table_3a',
#  'observed_peak_demand', 'table_3a_transposed', 'table_5', 'generation',
#  'table_6', 'connection_interest', 'infrastructure_projects', 'projects', 'cim']

Available LTDS datasets:
['table_1', 'circuit_data', 'table_2a', 'transformer_2w', 'table_2b', 'transformer_3w', 'table_3a', 'observed_demand', 'observed_peak_demand', 'table_3a_transposed', 'table_3b', 'forecast_demand', 'table_4a', 'fault_level_3ph', 'table_4b', 'fault_level_earth', 'table_5', 'generation', 'table_6', 'connection_interest', 'table_7', 'restrictions', 'table_8', 'fault_data', 'projects', 'infrastructure_projects', 'cim']


In [3]:
# Fetch CIM data for Eastern Power Networks (EPN)
cim_data = ltds.get_cim(licencearea="EPN", limit=100)

print(f"Total CIM records available: {cim_data.total_count}")
print(f"Records fetched: {len(cim_data.records)}")

# Expected output:
# Total CIM records available: 1234
# Records fetched: 100

TypeError: UKPNClient.get_records() got an unexpected keyword argument 'licencearea'

In [ ]:
# Inspect the structure of CIM records
if cim_data.records:
    first_record = cim_data.records[0]
    print("CIM Record fields:")
    if first_record.fields:
        for key, value in first_record.fields.items():
            print(f"  {key}: {value}")

## 2. Load Transformer Data (Table 2a/2b)

For pandapower network models, we need transformer specifications. LTDS Table 2a contains 2-winding transformer data, and Table 2b contains 3-winding transformer data.

In [ ]:
# Fetch 2-winding transformer data
transformers_2w = ltds.get_table_2a(licence_area="EPN", limit=50)

print(f"2-Winding Transformers: {transformers_2w.total_count} total")
print(f"Fetched: {len(transformers_2w.records)} records")

# Expected output:
# 2-Winding Transformers: 456 total
# Fetched: 50 records

In [ ]:
# Fetch 3-winding transformer data
transformers_3w = ltds.get_table_2b(licence_area="EPN", limit=50)

print(f"3-Winding Transformers: {transformers_3w.total_count} total")
print(f"Fetched: {len(transformers_3w.records)} records")

# Expected output:
# 3-Winding Transformers: 78 total
# Fetched: 50 records

In [ ]:
# Inspect transformer record structure
if transformers_2w.records:
    print("Transformer 2W fields:")
    for key, value in transformers_2w.records[0].fields.items():
        print(f"  {key}: {value}")

## 3. Load Demand Data (Table 3a)

Table 3a contains observed peak demand data at primary substations, which we'll use as load values in our pandapower model.

In [ ]:
# Fetch observed peak demand data
demand_data = ltds.get_table_3a(licence_area="EPN", limit=100)

print(f"Demand records: {demand_data.total_count} total")
print(f"Fetched: {len(demand_data.records)} records")

# Expected output:
# Demand records: 567 total
# Fetched: 100 records

In [ ]:
# Inspect demand record structure
if demand_data.records:
    print("Demand (Table 3a) fields:")
    for key, value in demand_data.records[0].fields.items():
        print(f"  {key}: {value}")

## 4. Load Generation Data (Table 5)

Table 5 contains distributed generation capacity connected at each primary substation.

In [ ]:
# Fetch generation data
generation_data = ltds.get_table_5(licence_area="EPN", limit=100)

print(f"Generation records: {generation_data.total_count} total")
print(f"Fetched: {len(generation_data.records)} records")

# Filter for solar generation
solar_gen = ltds.get_table_5(licence_area="EPN", technology_type="Solar", limit=50)
print(f"\nSolar generation records: {solar_gen.total_count}")

## 5. Convert to pandas DataFrames

pandapower works well with pandas DataFrames. Let's convert our UKPN data.

In [ ]:
def records_to_dataframe(response):
    """Convert RecordListResponse to pandas DataFrame."""
    if not response.records:
        return pd.DataFrame()

    data = [record.fields for record in response.records if record.fields]
    return pd.DataFrame(data)


# Convert to DataFrames
df_transformers = records_to_dataframe(transformers_2w)
df_demand = records_to_dataframe(demand_data)
df_generation = records_to_dataframe(generation_data)

print(f"Transformers DataFrame: {df_transformers.shape}")
print(f"Demand DataFrame: {df_demand.shape}")
print(f"Generation DataFrame: {df_generation.shape}")

In [ ]:
# Preview transformer data
print("Transformer Data Preview:")
display(df_transformers.head())

In [ ]:
# Preview demand data
print("Demand Data Preview:")
display(df_demand.head())

## 6. Create a pandapower Network

Now let's create a simple pandapower network using the UKPN data. This example shows the general approach - you'll need to adapt field names based on the actual CIM data structure.

In [ ]:
import pandapower as pp

# Create an empty pandapower network
net = pp.create_empty_network(name="UKPN EPN Network")

print(f"Created network: {net.name}")

In [ ]:
# Example: Create buses from substation data
# Note: Adjust field names based on actual CIM data structure

# Create a 132kV bus (Grid Supply Point)
bus_gsp = pp.create_bus(net, vn_kv=132, name="GSP_132kV")

# Create a 33kV bus (Primary substation HV side)
bus_primary_hv = pp.create_bus(net, vn_kv=33, name="Primary_33kV")

# Create an 11kV bus (Primary substation LV side)
bus_primary_lv = pp.create_bus(net, vn_kv=11, name="Primary_11kV")

print(f"Created {len(net.bus)} buses")
display(net.bus)

In [ ]:
# Create an external grid connection at the GSP
pp.create_ext_grid(net, bus=bus_gsp, vm_pu=1.0, name="National Grid Connection")

print("External grid created")
display(net.ext_grid)

In [ ]:
# Example: Create a transformer using LTDS Table 2a data
# Typical UKPN 132/33kV Grid Transformer parameters

pp.create_transformer_from_parameters(
    net,
    hv_bus=bus_gsp,
    lv_bus=bus_primary_hv,
    sn_mva=60,  # Rated power in MVA
    vn_hv_kv=132,  # HV side voltage
    vn_lv_kv=33,  # LV side voltage
    vk_percent=12,  # Short-circuit voltage %
    vkr_percent=0.5,  # Real part of short-circuit voltage %
    pfe_kw=50,  # Iron losses in kW
    i0_percent=0.1,  # No-load current %
    name="Grid_TX_1",
)

print("Transformer created")
display(net.trafo)

In [ ]:
# Create a 33/11kV primary transformer
pp.create_transformer_from_parameters(
    net,
    hv_bus=bus_primary_hv,
    lv_bus=bus_primary_lv,
    sn_mva=20,
    vn_hv_kv=33,
    vn_lv_kv=11,
    vk_percent=10,
    vkr_percent=0.5,
    pfe_kw=30,
    i0_percent=0.1,
    name="Primary_TX_1",
)

print(f"Total transformers: {len(net.trafo)}")
display(net.trafo)

In [ ]:
# Add a load using LTDS Table 3a peak demand data
# Example: 15 MW peak demand at 0.95 power factor
import math

peak_demand_mw = 15  # From Table 3a observed peak demand
power_factor = 0.95

# Calculate reactive power from power factor

q_mvar = peak_demand_mw * math.tan(math.acos(power_factor))

pp.create_load(
    net, bus=bus_primary_lv, p_mw=peak_demand_mw, q_mvar=q_mvar, name="Primary_Load"
)

print(f"Load created: {peak_demand_mw} MW, {q_mvar:.2f} Mvar")
display(net.load)

In [ ]:
# Add distributed generation using LTDS Table 5 data
# Example: 5 MW solar PV connected at 11kV

pp.create_sgen(
    net,
    bus=bus_primary_lv,
    p_mw=5,
    q_mvar=0,  # Unity power factor for PV
    name="Solar_PV_1",
    type="PV",
)

print("Static generator (PV) created")
display(net.sgen)

## 7. Run Power Flow Analysis

With the network model built, we can run a power flow calculation.

In [ ]:
# Run power flow calculation
pp.runpp(net, algorithm="nr", calculate_voltage_angles=True)

print("Power flow converged!")
print("\nBus Results:")
display(net.res_bus)

In [ ]:
# Transformer loading results
print("Transformer Loading:")
display(
    net.res_trafo[["p_hv_mw", "q_hv_mvar", "p_lv_mw", "q_lv_mvar", "loading_percent"]]
)

In [ ]:
# External grid power flow
print("External Grid Power Flow:")
display(net.res_ext_grid)

## 8. Visualize the Network

pandapower provides visualization tools through pandapower.plotting.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import pandapower.plotting as plot

    # Create a simple plot
    plot.simple_plot(net, show_plot=True)
    plt.title("UKPN Network Model")
    plt.show()
except ImportError:
    print("Plotting requires matplotlib. Install with: pip install matplotlib")
except Exception as e:
    print(f"Plotting not available: {e}")
    print("\nNetwork summary instead:")
    print(net)

## 9. Building a Larger Network from LTDS Data

Here's a function template to automate network creation from LTDS data.

In [ ]:
def create_network_from_ltds(licence_area: str = "EPN", limit: int = 50):
    """
    Create a pandapower network from UKPN LTDS data.

    Args:
        licence_area: UKPN licence area ('EPN', 'SPN', 'LPN')
        limit: Maximum records to fetch per dataset

    Returns:
        pandapower network object
    """
    # Fetch LTDS data
    transformers = ltds.get_table_2a(licence_area=licence_area, limit=limit)
    demand = ltds.get_table_3a(licence_area=licence_area, limit=limit)
    generation = ltds.get_table_5(licence_area=licence_area, limit=limit)

    # Create network
    net = pp.create_empty_network(name=f"UKPN {licence_area} Network")

    # Track created buses by substation name
    bus_map = {}  # NOQA F841

    # TODO: Implement actual mapping logic based on CIM data structure
    # This would parse transformer records to create buses and transformers
    # Then add loads from demand data and generators from generation data

    print(f"Network created for {licence_area}")
    print(f"  Transformer records: {transformers.total_count}")
    print(f"  Demand records: {demand.total_count}")
    print(f"  Generation records: {generation.total_count}")

    return net


# Example usage
net_epn = create_network_from_ltds("EPN", limit=20)

## 10. Export Network Data

You can export the pandapower network to various formats for further analysis.

In [ ]:
from pathlib import Path

save_dir = None  # Set to a directory (e.g. "exports") to enable writing files.

# Export to JSON (optional)
if save_dir:
    output_dir = Path(save_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    json_file = output_dir / "ukpn_network.json"
    pp.to_json(net, str(json_file))
    print(f"Network exported to {json_file}")
else:
    print("JSON export skipped; set save_dir to enable writing.")

# Export to Excel (optional)
if save_dir:
    output_dir = Path(save_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    xlsx_file = output_dir / "ukpn_network.xlsx"
    try:
        pp.to_excel(net, str(xlsx_file))
        print(f"Network exported to {xlsx_file}")
    except Exception as e:
        print(f"Excel export requires openpyxl: {e}")
else:
    print("Excel export skipped; set save_dir to enable writing.")

In [ ]:
# Also export the raw LTDS data for reference (optional)
from pathlib import Path

csv_data = ltds.export("table_2a", format="csv")

save_dir = None  # Set to a directory (e.g. "exports") to enable writing files.
if save_dir:
    output_file = Path(save_dir) / "ltds_transformers.csv"
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "wb") as f:
        f.write(csv_data)
    print(f"LTDS transformer data exported to {output_file}")
else:
    print("CSV export skipped; set save_dir to enable writing.")

## Summary

This tutorial demonstrated:

1. **Fetching LTDS CIM data** using `ltds.get_cim()`
2. **Loading transformer specifications** from Table 2a/2b
3. **Loading demand data** from Table 3a
4. **Loading generation data** from Table 5
5. **Converting to pandas DataFrames** for analysis
6. **Creating pandapower network elements** (buses, transformers, loads, generators)
7. **Running power flow analysis**
8. **Visualizing and exporting** the network

## Next Steps

- Explore the full CIM data structure to map all network elements
- Combine with time series powerflow data for dynamic studies
- Use DFES headroom data for scenario analysis
- Implement automated network topology discovery

## Resources

- [pandapower Documentation](https://pandapower.readthedocs.io/)
- [UKPN Open Data Portal](https://ukpowernetworks.opendatasoft.com/)
- [IEC CIM Standards](https://www.iec.ch/)